# Davinci — Tiny Overfit Diagnostic

Verify that the model, optimizer, and loss can learn a deliberately tiny fixed synthetic batch. This is a plumbing diagnostic only—not a corpus test, capability benchmark, or evidence of full-model readiness.

In [ ]:
import torch, json, sys
sys.path.insert(0,'/content/Davinci')
from src.davinci import DavinciConfig, DavinciForCausalLM
device='cuda' if torch.cuda.is_available() else 'cpu'; print('device:',device)
torch.manual_seed(7)


In [ ]:
cfg=DavinciConfig(vocab_size=128,hidden_size=64,num_hidden_layers=2,num_attention_heads=4,num_key_value_heads=2,head_dim=16,intermediate_size=128,local_window=32,global_every_n_layers=2,tie_word_embeddings=True,dropout=0.0)
model=DavinciForCausalLM(cfg).to(device); opt=torch.optim.AdamW(model.parameters(),lr=2e-3)
fixed=torch.randint(0,cfg.vocab_size,(4,32),device=device); labels=fixed.clone()
def loss_eval():
 model.eval()
 with torch.no_grad(): return float(model(fixed,labels=labels)['loss'])
initial=loss_eval(); model.train(); history=[]
for step in range(100):
 opt.zero_grad(set_to_none=True); out=model(fixed,labels=labels); loss=out['loss']; loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step(); history.append(float(loss.detach()))
final=loss_eval(); result={'initial_loss':initial,'final_loss':final,'steps':100,'reduced':final<initial}
print(json.dumps(result,indent=2))
if not final<initial: raise RuntimeError('Tiny fixed-batch overfit diagnostic did not reduce loss; inspect model/optimizer/precision.')


## Interpretation

Passing indicates basic gradient/optimizer flow on this synthetic batch. It does not verify tokenizer, real corpus, masking, GPU scaling, or generalization.